# 🛡️ Agent Engineering Challenge

**Duration:** ~60 minutes &nbsp;|&nbsp; **Solo build (peer collaboration encouraged)** &nbsp;|&nbsp; **Claude Messages API**

---

## What You're Building

An AI agent that automates RFP (Request for Proposal) responses for a cybersecurity vendor. Your agent will:

1. **Parse** a questionnaire into categorized questions
2. **Retrieve** relevant source material from a knowledge base
3. **Draft** polished, cited answers
4. **Review** answers for cross-question consistency
5. **Export** structured JSON output

The questionnaire in this notebook is what you'll develop and test against. At the end of the session you'll get a **surprise RFP** to point your agent at — so the goal isn't to hardcode the questions below, it's to build something flexible and robust enough to handle questions you haven't seen yet.

This is a solo build, but you're strongly encouraged to bounce ideas off the people around you — sketch architectures together, compare tool designs, debug each other's tool-use loops. Each person ships their own agent.

### Design Principles to Keep in Mind

- **Generality over specificity.** Tool design, prompts, and parsing should not assume the exact questions below.
- **Failure modes matter.** What happens when the KB doesn't have the answer? When a question spans two categories? When the prospect asks something off-script?
- **Consistency is the real quality bar.** Cross-question contradictions are the #1 failure mode of automated RFP responses — your review step is what separates a toy from something deployable.

### Build Something Worth Showing Off

There's no formal demo and no grading. But there's a room full of smart people who'd love to see something interesting. Past Partner Basecamp builds we've loved: RFP deck builders, robust eval harnesses, analysis UIs to visualize answer accuracy, search UIs that plug into any RFP, agents that route by category to specialized subagents. If you ship the baseline early, spend the remaining time building the thing **you** want to show off.

### Suggested Time Allocation

| Phase | Duration | Focus |
|-------|----------|-------|
| Setup & Planning | 5 min | Verify API, sketch architecture, identify your robustness bets |
| Build | 45 min | Agent loop, prompt design, testing on the included questionnaire |
| Surprise RFP | 5–8 min | Point your agent at the unseen RFP — see what holds up |

Extra time? Push harder on robustness, add tooling, or build the show-off feature you've been sitting on.

---

## Part 0: Environment Setup

Run the cells below to install dependencies and verify your API connection.

In [1]:
# Install dependencies
!pip install anthropic --upgrade -q

print("✓ Dependencies installed")

✓ Dependencies installed


In [2]:
import os
import json
from typing import Optional

import anthropic
import dotenv

# === API KEY SETUP ===
dotenv.load_dotenv()
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
DEFAULT_MODEL = "claude-sonnet-4-6"

if not ANTHROPIC_API_KEY:
    raise ValueError(
        "❌ ANTHROPIC_API_KEY not found.\n"
        "Set it via: export ANTHROPIC_API_KEY='your-key-here'\n"
        "Or paste directly in the cell above."
    )

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print("✓ Anthropic client initialized")

✓ Anthropic client initialized


In [3]:
# Verify API connectivity
try:
    response = client.messages.create(
        model=DEFAULT_MODEL,
        max_tokens=20,
        messages=[{"role": "user", "content": "Say 'ready' and nothing else."}]
    )
    print(f"✓ API connection verified: {response.content[0].text}")
    print(f"  Model: {response.model}")
except Exception as e:
    print(f"❌ API connection failed: {e}")

✓ API connection verified: ready
  Model: claude-sonnet-4-6


---

## Part 1: The Brief

### The Situation

**Helios Security** is a mid-market cybersecurity vendor selling endpoint protection, SIEM, and managed detection & response. Their sales team responds to **40+ RFPs per quarter**, each containing 50–200 questions spanning technical architecture, compliance certifications, pricing models, and company background.

Today, each RFP takes a solutions engineer **6–8 hours** to complete. The process is manual: hunt through a Confluence wiki of past proposals, cross-reference product documentation, and copy-paste answers into a spreadsheet — adjusting tone and specificity for each prospect. Answers frequently **contradict each other** across questions because no one reviews the full response holistically.

Helios wants an AI agent that takes in an RFP questionnaire and produces a draft response in minutes — decomposing questions, retrieving relevant source material, synthesizing polished answers, and flagging anything the human needs to review.

**The goal:** Cut first-draft time from 8 hours to under 15 minutes and eliminate cross-answer inconsistencies.

### The Agent Pipeline

```
┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐
│  PARSE   │───▶│ RETRIEVE │───▶│  DRAFT   │───▶│  REVIEW  │───▶│  EXPORT  │
│          │    │          │    │          │    │          │    │          │
│ Break    │    │ Search   │    │ Generate │    │ Check    │    │ Return   │
│ into Qs  │    │ mock KB  │    │ answers  │    │ for      │    │ struct.  │
│ + tag    │    │ via tool │    │ w/ cites │    │ contra-  │    │ JSON     │
│ category │    │          │    │          │    │ dictions │    │          │
└──────────┘    └──────────┘    └──────────┘    └──────────┘    └──────────┘
```

### Illustrative RFP Questions

The kinds of questions Helios sees in real RFPs span multiple categories — some have crisp answers in the KB, some don't, and some span two categories at once. Examples of the shape (these are **not** the exact questions you'll build against — those are in Part 5, and the surprise RFP at the end will be different again):

| ID | Category | Question |
|----|----------|----------|
| Ex1 | Technical | What is your incident response SLA for a confirmed P1 event? How is escalation handled outside business hours? |
| Ex2 | Compliance | Describe your sub-processor management process and how you notify customers of changes. |
| Ex3 | Pricing | What pricing flexibility is available for multi-year commitments, and how are price increases capped at renewal? |
| Ex4 | Company-Info | What is your customer support model? Describe tiers, channels, and average response times. |
| Ex5 | Technical + Compliance | How does your platform support customer-controlled key management, and which regions are FIPS 140-2 validated? |

Some of these are well-covered by the knowledge base. Some aren't. Your agent should handle both gracefully.

---

## Part 2: Mock Knowledge Base (Pre-Built)

Your agent needs data to retrieve from. Below is a pre-built mock knowledge base with realistic Helios Security content. **You can extend this with additional entries**, but the baseline is ready to use.

In [4]:
# ============================================================
# MOCK KNOWLEDGE BASE
# Pre-populated with realistic Helios Security content.
# Extend with additional entries if you want richer agent behavior.
# ============================================================

KNOWLEDGE_BASE = {
    "threat_detection": {
        "source": "Helios Platform Architecture Doc v4.2",
        "content": (
            "Helios Sentinel uses a multi-layered detection engine combining "
            "signature-based matching, behavioral analysis, and ML-driven anomaly detection. "
            "Data sources include endpoint telemetry (process events, file system changes, "
            "network connections), cloud workload logs (AWS CloudTrail, Azure Activity Log, "
            "GCP Audit Log), network flow data (NetFlow v9/IPFIX), and email gateway events. "
            "Average detection-to-alert latency is 2.3 seconds for signature matches and "
            "18 seconds for behavioral detections. Our SIEM correlation engine processes "
            "up to 50,000 events per second per tenant."
        ),
        "tags": ["technical", "detection", "latency", "architecture"]
    },
    "compliance_certs": {
        "source": "Helios Compliance & Certifications Register 2025",
        "content": (
            "Current certifications: SOC 2 Type II (audited December 2024 by Deloitte), "
            "ISO 27001:2022 (certified March 2024 by BSI), FedRAMP Moderate (authorized "
            "June 2024, sponsored by DHS), HIPAA (BAA available, last assessment October 2024), "
            "PCI DSS v4.0 Level 1 Service Provider (validated September 2024 by Coalfire). "
            "StateRAMP authorized (January 2025). All certifications maintained on continuous "
            "monitoring basis with quarterly internal audits."
        ),
        "tags": ["compliance", "certifications", "audit", "soc2", "fedramp"]
    },
    "pricing_model": {
        "source": "Helios Commercial Pricing Sheet Q1 2025",
        "content": (
            "Endpoint Protection Platform (EPP+EDR bundle): "
            "500 endpoints: $18/seat/month ($108,000/year). "
            "1,000 endpoints: $15/seat/month ($180,000/year) — 17% volume discount. "
            "5,000 endpoints: $11/seat/month ($660,000/year) — 39% volume discount. "
            "Minimum contract term: 12 months. Multi-year discounts: 2-year = additional 5%, "
            "3-year = additional 10%. SIEM add-on: +$6/seat/month. "
            "MDR add-on: +$12/seat/month. All pricing excludes professional services."
        ),
        "tags": ["pricing", "commercial", "discount", "contract"]
    },
    "financial_services_customers": {
        "source": "Helios Customer Success — Vertical Report 2024",
        "content": (
            "Helios currently serves 47 customers in financial services, including "
            "12 banks, 8 insurance carriers, 15 asset management firms, and 12 fintech companies. "
            "Reference accounts (approved for external use): "
            "1) Meridian National Bank — 3,200 endpoints, EPP+EDR+SIEM, deployed since 2022. "
            "2) Crestview Capital Partners — 850 endpoints, EPP+MDR, deployed since 2023. "
            "3) Apex Insurance Group — 5,100 endpoints, full platform, deployed since 2021. "
            "Average NPS in financial services vertical: 72."
        ),
        "tags": ["company-info", "customers", "financial-services", "references"]
    },
    "data_residency_eu": {
        "source": "Helios Data Sovereignty & Privacy Whitepaper v3.1",
        "content": (
            "Helios supports full EU data residency through dedicated infrastructure in "
            "Frankfurt (AWS eu-central-1) and Dublin (AWS eu-west-1). Customer data never "
            "leaves the selected region. Encryption at rest: AES-256-GCM with customer-managed "
            "keys (AWS KMS or BYOK). Encryption in transit: TLS 1.3 for all API and agent "
            "communications, with certificate pinning for endpoint agents. "
            "GDPR Data Processing Agreement (DPA) included in all EU contracts. "
            "Annual third-party penetration testing by NCC Group. "
            "Data retention: configurable per tenant, default 90 days for raw telemetry, "
            "13 months for aggregated alerts."
        ),
        "tags": ["technical", "compliance", "data-residency", "eu", "encryption", "gdpr"]
    },
    "past_rfp_detection_answer": {
        "source": "Acme Corp RFP Response — March 2024",
        "content": (
            "Q: Describe your real-time threat detection capabilities. "
            "A: Helios Sentinel provides sub-3-second detection for known threat patterns "
            "and under 20 seconds for behavioral anomalies. Our detection engine ingests "
            "endpoint telemetry, network flows, cloud audit logs, and email events. "
            "The SIEM correlation engine handles 50K EPS per tenant. "
            "We maintain a 99.7% true positive rate on our top 100 detection rules, "
            "validated quarterly against MITRE ATT&CK framework."
        ),
        "tags": ["technical", "detection", "past-rfp"]
    },
    "past_rfp_compliance_answer": {
        "source": "NovaTech RFP Response — July 2024",
        "content": (
            "Q: What compliance certifications do you hold? "
            "A: Helios holds SOC 2 Type II, ISO 27001, FedRAMP Moderate, PCI DSS v4.0, "
            "and HIPAA compliance. All certifications are actively maintained with "
            "continuous monitoring. We provide audit reports upon request under NDA. "
            "Our security team of 14 full-time engineers manages compliance programs."
        ),
        "tags": ["compliance", "certifications", "past-rfp"]
    }
}


def search_knowledge_base(query: str, category: Optional[str] = None) -> list[dict]:
    """
    Search the mock knowledge base.
    Returns matching entries ranked by keyword overlap.
    """
    query_terms = set(query.lower().split())
    results = []

    for entry_id, entry in KNOWLEDGE_BASE.items():
        # Score by keyword overlap with content + tags
        entry_text = (entry["content"] + " " + " ".join(entry["tags"])).lower()
        overlap = len(query_terms & set(entry_text.split()))

        # Boost if category matches a tag
        if category and category.lower() in [t.lower() for t in entry["tags"]]:
            overlap += 5

        if overlap > 0:
            results.append({
                "id": entry_id,
                "source": entry["source"],
                "content": entry["content"],
                "relevance_score": overlap
            })

    # Sort by relevance, return top 3
    results.sort(key=lambda x: x["relevance_score"], reverse=True)
    return results[:3]


# Quick test
test_results = search_knowledge_base("threat detection latency", category="technical")
print(f"✓ Knowledge base loaded ({len(KNOWLEDGE_BASE)} entries)")
print(f"  Test search 'threat detection latency': {len(test_results)} results")
print(f"  Top result: {test_results[0]['source']}")

✓ Knowledge base loaded (7 entries)
  Test search 'threat detection latency': 3 results
  Top result: Helios Platform Architecture Doc v4.2


---

## Part 3: Tool Definition

Define the `search_kb` tool so Claude can retrieve information from the knowledge base during its agent loop.

In [5]:
# Tool definition for the Claude API
SEARCH_KB_TOOL = {
    "name": "search_kb",
    "description": (
        "Search the Helios Security knowledge base for information relevant to "
        "answering an RFP question. Returns up to 3 matching documents with source "
        "attribution. Use this to find product docs, past proposal answers, compliance "
        "records, and pricing information."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query — use keywords from the RFP question"
            },
            "category": {
                "type": "string",
                "enum": ["technical", "compliance", "pricing", "company-info"],
                "description": "Optional category filter to narrow results"
            }
        },
        "required": ["query"]
    }
}


def handle_tool_call(tool_name: str, tool_input: dict) -> str:
    """Execute a tool call and return the result as a string."""
    if tool_name == "search_kb":
        results = search_knowledge_base(
            query=tool_input["query"],
            category=tool_input.get("category")
        )
        return json.dumps(results, indent=2)
    else:
        return json.dumps({"error": f"Unknown tool: {tool_name}"})


print("✓ Tool definition ready")

✓ Tool definition ready


---

## Part 4: Level 0 Agent (Working Baseline)

Below is a working agent that handles a **single question** end-to-end. It:
- Sends the question to Claude with the `search_kb` tool available
- Handles the tool use loop (Claude calls the tool → we execute it → send results back)
- Returns a structured answer

**Run this to confirm the agent loop works**, then extend it in Part 5.

In [6]:
SYSTEM_PROMPT = """You are HeliBot, an AI assistant helping Helios Security respond to RFP questionnaires.

For each question, you must:
1. Use the search_kb tool to find relevant source material
2. Draft a professional, detailed answer grounded in the retrieved sources
3. Cite your sources by name
4. If the knowledge base doesn't contain enough information, flag the answer as low-confidence

Return your answer as JSON with this structure:
{
    "question_id": "Q1",
    "category": "technical",
    "answer": "Your drafted answer here...",
    "sources": ["Source Name 1", "Source Name 2"],
    "confidence": "high" | "medium" | "low",
    "flags": ["any concerns or notes for human review"]
}

Be specific, professional, and concise. Use concrete numbers from the source material."""


def answer_single_question(
    question_id: str,
    question_text: str,
    category: str,
    model: str = DEFAULT_MODEL,
) -> dict:
    """
    Level 0 agent: answers a single RFP question with tool use.
    Handles the full tool use loop.
    """
    messages = [
        {
            "role": "user",
            "content": (
                f"Answer this RFP question.\n\n"
                f"Question ID: {question_id}\n"
                f"Category: {category}\n"
                f"Question: {question_text}\n\n"
                f"Search the knowledge base for relevant information, then draft your answer."
            )
        }
    ]

    # Agent loop: keep going until Claude stops calling tools
    max_turns = 5  # Safety limit
    for turn in range(max_turns):
        response = client.messages.create(
            model=model,
            max_tokens=2048,
            system=SYSTEM_PROMPT,
            messages=messages,
            tools=[SEARCH_KB_TOOL],
        )

        # If Claude is done (no more tool calls), extract the answer
        if response.stop_reason == "end_turn":
            # Find the text block with the JSON answer
            for block in response.content:
                if block.type == "text":
                    try:
                        # Try to parse JSON from the response
                        text = block.text
                        # Handle markdown code blocks
                        if "```json" in text:
                            text = text.split("```json")[1].split("```")[0]
                        elif "```" in text:
                            text = text.split("```")[1].split("```")[0]
                        return json.loads(text.strip())
                    except json.JSONDecodeError:
                        return {"raw_response": block.text, "parse_error": True}
            break

        # If Claude wants to use a tool, execute it and continue
        if response.stop_reason == "tool_use":
            # Add assistant's response to message history
            messages.append({"role": "assistant", "content": response.content})

            # Execute each tool call and collect results
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = handle_tool_call(block.name, block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                    })

            messages.append({"role": "user", "content": tool_results})

    return {"error": "Max turns reached without completing"}


print("✓ Level 0 agent defined")

✓ Level 0 agent defined


In [7]:
# Test: Answer Q1
print("Answering Q1 (threat detection)...")
result = answer_single_question(
    question_id="Q1",
    question_text=(
        "Describe your platform's approach to real-time threat detection. "
        "What data sources are ingested, and what is the average detection-to-alert latency?"
    ),
    category="technical",
)

print(json.dumps(result, indent=2))

Answering Q1 (threat detection)...
{
  "question_id": "Q1",
  "category": "technical",
  "answer": "Helios Sentinel employs a multi-layered, real-time threat detection engine that combines three complementary detection methodologies: (1) signature-based matching for known threat patterns, (2) behavioral analysis for anomalous activity, and (3) ML-driven anomaly detection for novel or low-signal threats. All three layers operate continuously and in parallel, feeding a centralized SIEM correlation engine capable of processing up to 50,000 events per second (EPS) per tenant.\n\n**Data Sources Ingested:**\nHelios Sentinel ingests a broad range of telemetry across the modern enterprise attack surface, including:\n- **Endpoint Telemetry:** Process events, file system changes, and network connections from managed endpoints.\n- **Cloud Workload Logs:** AWS CloudTrail, Azure Activity Log, and GCP Audit Log for multi-cloud visibility.\n- **Network Flow Data:** NetFlow v9 and IPFIX for network tr

---

## Part 5: 🛠️ YOUR TASK — Multi-Question Agent

The Level 0 agent handles one question at a time. Your job is to build a **full pipeline** that:

1. **Accepts a list of RFP questions** (the full questionnaire)
2. **Processes each question** through the agent loop
3. **Collects all answers** into a structured response
4. **Exports as JSON** with all required fields

### Approaches to Consider

- **Sequential:** Process questions one by one using `answer_single_question()`. Simple, reliable, but slower.
- **Batch prompt:** Send all questions to Claude in a single prompt and let it call the tool multiple times. Faster, but harder to control.
- **Hybrid:** Parse and categorize first, then process by category to reuse retrieved context.

### Starter Questions

In [8]:
# RFP Questionnaire
RFP_QUESTIONS = [
    {
        "id": "Q1",
        "category": "technical",
        "text": (
            "Describe your platform's approach to real-time threat detection. "
            "What data sources are ingested, and what is the average detection-to-alert latency?"
        ),
    },
    {
        "id": "Q2",
        "category": "compliance",
        "text": (
            "List all compliance certifications your organization currently holds "
            "(SOC 2, ISO 27001, FedRAMP, etc.) and the date of most recent audit for each."
        ),
    },
    {
        "id": "Q3",
        "category": "pricing",
        "text": (
            "Provide per-seat pricing for 500, 1,000, and 5,000 endpoints. "
            "Are volume discounts available? Is there a minimum contract term?"
        ),
    },
    {
        "id": "Q4",
        "category": "company-info",
        "text": (
            "How many customers do you currently serve in the financial services vertical? "
            "Provide 2–3 reference accounts."
        ),
    },
    {
        "id": "Q5",
        "category": "technical",
        "text": (
            "How does your platform handle data residency requirements for customers "
            "operating in the EU? Describe encryption at rest and in transit."
        ),
    },
]

In [9]:
# ============================================================
# 🛠️ YOUR CODE: Build the multi-question agent
# ============================================================
#
# Process all RFP_QUESTIONS and collect structured responses.
#
# Your output should be a list of answer objects, each containing:
#   - question_id
#   - category
#   - answer
#   - sources
#   - confidence (high / medium / low)
#   - flags (list of concerns for human review)
#
# HINT: The simplest approach is a loop over answer_single_question().
# More advanced: modify the system prompt to handle batches.
# ============================================================

def process_rfp(questions: list[dict]) -> list[dict]:
    """Process a full RFP questionnaire and return structured answers."""
    answers = []

    for q in questions:
        q_id = q.get("id", "?")
        category = q.get("category", "")
        text = q.get("text", "")
        print(f"  → {q_id} ({category})...")

        try:
            result = answer_single_question(
                question_id=q_id,
                question_text=text,
                category=category,
            )
        except Exception as e:
            # Don't let one failure kill the whole RFP — record it and move on.
            result = {
                "question_id": q_id,
                "category": category,
                "answer": "",
                "sources": [],
                "confidence": "low",
                "flags": [f"agent_error: {type(e).__name__}: {e}"],
            }

        # Backfill identifiers in case the model omitted them, and ensure the
        # required keys exist so downstream review/eval/export steps don't break.
        result.setdefault("question_id", q_id)
        result.setdefault("category", category)
        result.setdefault("answer", "")
        result.setdefault("sources", [])
        result.setdefault("confidence", "low")
        result.setdefault("flags", [])

        # If JSON parsing failed in the Level 0 agent, surface that as a flag
        # rather than silently passing a malformed record through the pipeline.
        if result.get("parse_error"):
            result["flags"].append("parse_error: model output was not valid JSON")
            result["confidence"] = "low"

        answers.append(result)

    return answers


# Run it
print("Processing RFP...")
all_answers = process_rfp(RFP_QUESTIONS)

# Display results
if all_answers:
    for ans in all_answers:
        q_id = ans.get('question_id', '?')
        conf = ans.get('confidence', '?')
        print(f"  {q_id}: confidence={conf}, sources={len(ans.get('sources', []))}")
    print(f"\n✓ Processed {len(all_answers)} questions")
else:
    print("❌ No answers returned — implement process_rfp() above")


Processing RFP...
  → Q1 (technical)...
  → Q2 (compliance)...
  → Q3 (pricing)...
  → Q4 (company-info)...
  → Q5 (technical)...
  Q1: confidence=high, sources=2
  Q2: confidence=high, sources=2
  Q3: confidence=high, sources=1
  Q4: confidence=high, sources=1
  Q5: confidence=high, sources=2

✓ Processed 5 questions


---

## Part 6: 🛠️ YOUR TASK — Consistency Review Step

The customer specifically asked for **cross-answer consistency checking**. Answers drafted individually often contradict each other (e.g., Q2 says "FedRAMP authorized June 2024" but Q5 says "FedRAMP certified in 2023").

Build a review step that:
1. Takes all drafted answers as input
2. Identifies potential contradictions or inconsistencies
3. Flags issues for human review

**This is what separates a good agent from a great one.** In real deployments, consistency is the #1 quality issue in automated RFP responses.

In [10]:
# ============================================================
# 🛠️ YOUR CODE: Consistency review step
# ============================================================
#
# Send all answers to Claude for cross-answer review.
# Look for: contradictions, inconsistent data points, tone mismatches.
#
# HINT: Format all answers as a single prompt and ask Claude to
# identify any inconsistencies. No tool use needed for this step.
# ============================================================

REVIEW_SYSTEM_PROMPT = """You are a senior RFP reviewer for Helios Security. You are auditing a draft set of answers \
written by a junior teammate for cross-question consistency before the response goes to the prospect.

You are looking for problems a prospect or their procurement/legal team would notice:

1. **Factual contradictions** — same fact stated differently across answers (e.g. one answer says FedRAMP was \
authorized June 2024, another says 2023; one quotes 99.7% TPR, another 99.5%; pricing tiers that don't reconcile).
2. **Inconsistent specificity** — one answer cites a hard number, another hand-waves the same metric.
3. **Tone / register drift** — answers that sound like marketing fluff next to answers that are crisp and technical.
4. **Source / citation gaps** — answers that make strong claims without a cited source, or that cite sources that \
don't appear elsewhere in the response.
5. **Confidence calibration** — answers marked high-confidence that are actually thin, or low-confidence answers \
that are well-sourced and should be upgraded.

Return ONLY a JSON object — no prose, no markdown fences — with this exact shape:

{
  "ready_to_send": true | false,
  "summary": "1–2 sentence overall verdict",
  "issues": [
    {
      "severity": "high" | "medium" | "low",
      "type": "contradiction" | "specificity" | "tone" | "sourcing" | "confidence" | "other",
      "questions": ["Q2", "Q5"],
      "description": "Concrete description of the issue, quoting the conflicting text.",
      "recommendation": "What the human reviewer should do to resolve it."
    }
  ]
}

If there are no issues, return an empty `issues` list and set `ready_to_send` to true. Be specific — vague \
flags like 'check Q3' are not useful. Quote the exact conflicting phrases."""


def _format_answers_for_review(answers: list[dict]) -> str:
    """Render the answer set as a compact, reviewer-friendly block."""
    lines = []
    for ans in answers:
        lines.append(f"--- {ans.get('question_id', '?')} ({ans.get('category', '?')}) ---")
        lines.append(f"Answer: {ans.get('answer', '')}")
        lines.append(f"Sources: {', '.join(ans.get('sources', [])) or '(none cited)'}")
        lines.append(f"Confidence: {ans.get('confidence', '?')}")
        existing_flags = ans.get("flags", [])
        if existing_flags:
            lines.append(f"Existing flags: {existing_flags}")
        lines.append("")
    return "\n".join(lines)


def review_answers(answers: list[dict], model: str = DEFAULT_MODEL) -> dict:
    """
    Review all drafted answers for cross-question consistency.
    Returns a review report with any flagged issues.
    """
    if not answers:
        return {"ready_to_send": False, "summary": "No answers to review.", "issues": []}

    user_prompt = (
        "Review the following RFP answer set for cross-question consistency.\n\n"
        f"{_format_answers_for_review(answers)}\n"
        "Return your review as the JSON object specified in the system prompt."
    )

    # No tools — this is a pure reasoning pass over the drafted answers.
    # max_tokens=4096: a thorough audit on 5+ answers can easily run past 2k
    # tokens of issue prose; truncation mid-JSON breaks the downstream parser.
    response = client.messages.create(
        model=model,
        max_tokens=4096,
        system=REVIEW_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_prompt}],
    )

    text = "".join(block.text for block in response.content if block.type == "text").strip()

    # The system prompt asks for raw JSON, but strip code fences defensively in case
    # the model wraps the output anyway — same pattern as answer_single_question.
    if "```json" in text:
        text = text.split("```json", 1)[1].split("```", 1)[0]
    elif text.startswith("```"):
        text = text.split("```", 2)[1]
        if text.startswith("json"):
            text = text[4:]

    try:
        return json.loads(text.strip())
    except json.JSONDecodeError as e:
        return {
            "ready_to_send": False,
            "summary": "Review step returned non-JSON output — needs manual inspection.",
            "issues": [],
            "parse_error": str(e),
            "raw_response": text,
        }


# Run review on your answers
if all_answers:
    print("Running consistency review...")
    review = review_answers(all_answers)
    print(json.dumps(review, indent=2))
else:
    print("⚠️ No answers to review yet — complete Part 5 first")


Running consistency review...
{
  "ready_to_send": false,
  "summary": "The draft contains several cross-question consistency issues that a procurement or legal reviewer would likely flag, including a compliance date discrepancy, an unreconciled TPR metric, sourcing from prior client RFP responses, and one arithmetic error in the pricing table. These must be resolved before submission.",
  "issues": [
    {
      "severity": "high",
      "type": "contradiction",
      "questions": [
        "Q3"
      ],
      "description": "The 1,000-seat annual total is arithmetically wrong. At $15/seat/month \u00d7 1,000 seats \u00d7 12 months = $180,000/yr, which is correct. However, the stated volume discount of '17% savings' is inconsistent: ($18 - $15) / $18 = 16.67%, which rounds to 17% \u2014 that is technically acceptable. BUT the 5,000-seat row: $11/seat/month \u00d7 5,000 \u00d7 12 = $660,000/yr, and the stated discount is '39% savings' vs the baseline $18. ($18 - $11) / $18 = 38.89%, rou

---

## Part 7: Export Final Output

Package your agent's output as clean JSON — the deliverable Helios would actually receive.

In [11]:
# ============================================================
# Part 7: Final Deliverables
# ------------------------------------------------------------
# Three artifacts from one run:
#   1. internal_report.md    — markdown triage report for the SE.
#   2. client_response.html  — self-contained CxO dashboard for the
#                              prospect (no internal-only fields).
#   3. rfp_response.json     — structured machine-readable record
#                              for downstream tooling.
#
# Output is written to ./rfp_output/. We don't echo the rendered
# documents back into the notebook — open the files directly to view.
# ============================================================

import html as _html
import pathlib
import re
from collections import Counter
from datetime import date


RFP_NAME = "Sample RFP — Agent Engineering Challenge"
VENDOR_NAME = "Helios Security"

# Pull review from globals — Part 6 may not have been run yet, and
# a parse-failure review still counts as "ran" (we want to surface it).
_review = globals().get("review") if isinstance(globals().get("review"), dict) else None


# ------------------------------------------------------------
# Markdown → HTML helper for answer bodies
# ------------------------------------------------------------
# The drafted answers can use **bold**, "- " bullets, paragraphs, and
# pipe-style tables. Prefer the `markdown` package if installed; otherwise
# fall back to a built-in converter that covers what HeliBot emits.
def _md_is_table_separator(line: str) -> bool:
    """A markdown table's row 2 looks like '|---|:---:|---:|' (with optional outer pipes)."""
    stripped = line.strip()
    if "|" not in stripped:
        return False
    cells = [c.strip() for c in stripped.strip("|").split("|")]
    if not cells:
        return False
    return all(re.match(r"^:?-{3,}:?$", c) for c in cells if c)


def _md_parse_table_row(line: str) -> list[str]:
    line = line.strip()
    if line.startswith("|"):
        line = line[1:]
    if line.endswith("|"):
        line = line[:-1]
    return [c.strip() for c in line.split("|")]


try:
    import markdown as _md_lib

    def _answer_to_html(text: str) -> str:
        # `tables` is bundled in `extra`, but listing it explicitly documents intent.
        return _md_lib.markdown(
            text or "",
            extensions=["extra", "tables", "sane_lists"],
        )

except ImportError:
    def _answer_to_html(text: str) -> str:
        if not text:
            return ""
        escaped = _html.escape(text)
        # Inline formatting first: bold (greedy on **...**) then italics.
        escaped = re.sub(r"\*\*([^*\n]+)\*\*", r"<strong>\1</strong>", escaped)
        escaped = re.sub(r"(?<!\*)\*([^*\n]+)\*(?!\*)", r"<em>\1</em>", escaped)

        out_blocks = []
        for block in re.split(r"\n\s*\n", escaped.strip()):
            lines = [ln for ln in block.split("\n") if ln.strip()]
            if not lines:
                continue

            # Tables: 2+ lines and row 2 is a separator.
            if len(lines) >= 2 and _md_is_table_separator(lines[1]):
                headers = _md_parse_table_row(lines[0])
                data_rows = [_md_parse_table_row(ln) for ln in lines[2:]]
                thead = (
                    "<thead><tr>"
                    + "".join(f"<th>{h}</th>" for h in headers)
                    + "</tr></thead>"
                )
                tbody = (
                    "<tbody>"
                    + "".join(
                        "<tr>" + "".join(f"<td>{c}</td>" for c in row) + "</tr>"
                        for row in data_rows
                    )
                    + "</tbody>"
                )
                out_blocks.append(f"<table>{thead}{tbody}</table>")
                continue

            # Bullet lists: every non-empty line starts with -, *, or •.
            if all(re.match(r"^\s*[-*•]\s+", ln) for ln in lines):
                items = "".join(
                    f"<li>{re.sub(r'^\s*[-*•]\s+', '', ln)}</li>" for ln in lines
                )
                out_blocks.append(f"<ul>{items}</ul>")
                continue

            # Paragraph fallback: join lines with soft line breaks.
            out_blocks.append("<p>" + "<br>".join(lines) + "</p>")

        return "\n".join(out_blocks)


# Category icons + display names. Unknown categories fall back to a generic doc icon.
CATEGORY_META = {
    "technical": ("🔧", "Technical"),
    "compliance": ("🛡️", "Compliance & Security"),
    "pricing": ("💰", "Commercial & Pricing"),
    "company-info": ("🏢", "Company Information"),
}


# ------------------------------------------------------------
# Internal markdown report
# ------------------------------------------------------------
def build_internal_report(rfp_name, questions, answers, review):
    """Triage report for the human reviewer before the response ships."""
    lines = [f"# Internal Review Report — {rfp_name}", ""]
    lines.append(f"**Generated:** {date.today().isoformat()}  ")
    lines.append(f"**Model:** `{DEFAULT_MODEL}`  ")
    lines.append(f"**Knowledge base entries:** {len(KNOWLEDGE_BASE)}")
    lines.append("")

    lines.append("## RFP Overview")
    lines.append("")
    lines.append(f"- Total questions: **{len(questions)}**")
    cats = Counter(a.get("category", "uncategorized") for a in answers)
    if cats:
        cat_str = ", ".join(f"{c} ({n})" for c, n in sorted(cats.items()))
        lines.append(f"- Categories: {cat_str}")
    lines.append("")

    confs = Counter(a.get("confidence", "unknown") for a in answers)
    total = max(len(answers), 1)
    high, med, low = confs.get("high", 0), confs.get("medium", 0), confs.get("low", 0)
    sourced = sum(1 for a in answers if a.get("sources"))
    lines.append("## Draft Quality")
    lines.append("")
    lines.append(f"- High confidence: **{high}** ({high * 100 // total}%)")
    lines.append(f"- Medium confidence: **{med}** ({med * 100 // total}%)")
    lines.append(f"- Low confidence: **{low}** ({low * 100 // total}%)")
    lines.append(f"- Answers with cited sources: **{sourced} / {len(answers)}**")
    lines.append("")

    lines.append("## Per-Question Summary")
    lines.append("")
    lines.append("| ID | Category | Confidence | Sources | Flags |")
    lines.append("|----|----------|------------|---------|-------|")
    for a in answers:
        lines.append(
            f"| {a.get('question_id', '?')} "
            f"| {a.get('category', '?')} "
            f"| {a.get('confidence', '?')} "
            f"| {len(a.get('sources', []))} "
            f"| {len(a.get('flags', []))} |"
        )
    lines.append("")

    flagged = [a for a in answers if a.get("flags")]
    lines.append("## Items Flagged During Drafting")
    lines.append("")
    if not flagged:
        lines.append("_No per-question flags raised._")
        lines.append("")
    else:
        for a in flagged:
            lines.append(f"### {a.get('question_id')} ({a.get('category')})")
            lines.append("")
            for f in a["flags"]:
                lines.append(f"- {f}")
            lines.append("")

    lines.append("## Cross-Question Review")
    lines.append("")
    if not review:
        lines.append("_Review step was not run — re-run Part 6 before sending._")
        lines.append("")
    elif review.get("parse_error"):
        lines.append(
            f"⚠️ _Review step ran but returned malformed JSON_ "
            f"(`{review.get('parse_error')}`). Inspect the raw output manually."
        )
        lines.append("")
    else:
        ready = review.get("ready_to_send")
        summary = review.get("summary", "(no summary returned)")
        verdict = "✅ Ready to send" if ready else "⚠️ Needs human review"
        lines.append(f"**Verdict:** {verdict} — {summary}")
        lines.append("")
        issues = review.get("issues", [])
        if issues:
            lines.append("### Issues to resolve")
            lines.append("")
            for i, issue in enumerate(issues, 1):
                sev = issue.get("severity", "?").upper()
                itype = issue.get("type", "?")
                qs = ", ".join(issue.get("questions", [])) or "(unscoped)"
                lines.append(f"**{i}. [{sev}] {itype} — {qs}**")
                lines.append("")
                lines.append(f"- _Description:_ {issue.get('description', '')}")
                lines.append(f"- _Recommendation:_ {issue.get('recommendation', '')}")
                lines.append("")

    lines.append("## Human Reviewer Checklist")
    lines.append("")
    checklist = []
    if low > 0:
        checklist.append(
            f"- [ ] Verify the **{low}** low-confidence answer(s) against authoritative source material."
        )
    missing_sources = [a for a in answers if not a.get("sources")]
    if missing_sources:
        ids = ", ".join(a.get("question_id", "?") for a in missing_sources)
        checklist.append(f"- [ ] Add citations to answers missing a source: {ids}.")
    if review and review.get("issues"):
        checklist.append(
            f"- [ ] Resolve the **{len(review['issues'])}** cross-question issue(s) above."
        )
    if review and review.get("parse_error"):
        checklist.append("- [ ] Re-run the consistency review — output was malformed.")
    if flagged:
        ids = ", ".join(a.get("question_id", "?") for a in flagged)
        checklist.append(f"- [ ] Review per-question flags on: {ids}.")
    if not checklist:
        checklist.append("- [x] No outstanding issues detected — proof-read once and ship.")
    lines.extend(checklist)

    return "\n".join(lines).rstrip() + "\n"


# ------------------------------------------------------------
# Client-facing CxO dashboard (self-contained HTML)
# ------------------------------------------------------------
_DASHBOARD_CSS = """
:root {
  --bg: #f5f7fb;
  --surface: #ffffff;
  --surface-2: #eef1f7;
  --ink: #0d1b2a;
  --ink-soft: #3f4a5c;
  --muted: #7a8499;
  --accent: #1a4fd6;
  --accent-soft: #e6edfd;
  --accent-deep: #0b2a8a;
  --border: #e3e7ef;
  --shadow-sm: 0 1px 2px rgba(13,27,42,0.04);
  --shadow: 0 1px 3px rgba(13,27,42,0.06), 0 8px 24px rgba(13,27,42,0.06);
  --radius: 14px;
}
* { box-sizing: border-box; }
html, body { margin: 0; padding: 0; }
body {
  font-family: "Inter", -apple-system, BlinkMacSystemFont, "Segoe UI", Helvetica, Arial, sans-serif;
  color: var(--ink);
  background: var(--bg);
  line-height: 1.55;
  -webkit-font-smoothing: antialiased;
}
.container { max-width: 1120px; margin: 0 auto; padding: 0 32px; }

/* ---- Hero ---- */
.hero {
  background: radial-gradient(120% 140% at 0% 0%, #1a4fd6 0%, #0d1b2a 70%);
  color: #fff;
  padding: 56px 0 72px;
}
.hero-inner {
  display: flex;
  align-items: flex-end;
  justify-content: space-between;
  gap: 32px;
}
.hero h1 {
  font-size: 38px;
  margin: 4px 0 12px;
  font-weight: 700;
  letter-spacing: -0.025em;
  line-height: 1.15;
}
.eyebrow {
  font-size: 12px;
  text-transform: uppercase;
  letter-spacing: 0.12em;
  opacity: 0.72;
  font-weight: 600;
}
.hero .sub {
  font-size: 16px;
  opacity: 0.82;
  max-width: 620px;
  margin: 0;
}
.hero-meta {
  font-size: 13px;
  opacity: 0.78;
  text-align: right;
  white-space: nowrap;
}
.hero-meta strong { color: #fff; }

/* ---- KPI strip (pulled up over the hero) ---- */
.kpis {
  display: grid;
  grid-template-columns: repeat(4, 1fr);
  gap: 16px;
  margin: -36px 0 40px;
}
.kpi {
  background: var(--surface);
  border-radius: var(--radius);
  padding: 22px 24px;
  box-shadow: var(--shadow);
  border: 1px solid var(--border);
}
.kpi .label {
  font-size: 11px;
  text-transform: uppercase;
  letter-spacing: 0.08em;
  color: var(--muted);
  font-weight: 600;
  margin-bottom: 10px;
}
.kpi .value {
  font-size: 32px;
  font-weight: 700;
  letter-spacing: -0.02em;
  color: var(--ink);
  line-height: 1;
}
.kpi .hint { font-size: 12px; color: var(--ink-soft); margin-top: 6px; }

/* ---- Category section cards ---- */
.section {
  background: var(--surface);
  border: 1px solid var(--border);
  border-radius: var(--radius);
  padding: 28px 32px;
  margin-bottom: 20px;
  box-shadow: var(--shadow-sm);
}
.section-header {
  display: flex;
  align-items: center;
  gap: 14px;
  padding-bottom: 18px;
  margin-bottom: 22px;
  border-bottom: 1px solid var(--border);
}
.section-header .icon {
  width: 44px; height: 44px;
  border-radius: 10px;
  background: var(--accent-soft);
  color: var(--accent);
  display: flex; align-items: center; justify-content: center;
  font-size: 22px;
  flex-shrink: 0;
}
.section-header h2 {
  margin: 0;
  font-size: 20px;
  letter-spacing: -0.01em;
  font-weight: 700;
}
.section-header .count {
  margin-left: auto;
  font-size: 12px;
  color: var(--muted);
  text-transform: uppercase;
  letter-spacing: 0.06em;
  font-weight: 600;
}

/* ---- Q&A block ---- */
.qa { padding: 20px 0; border-top: 1px solid var(--border); }
.qa:first-of-type { border-top: 0; padding-top: 0; }
.qa-question {
  font-size: 16px;
  font-weight: 600;
  color: var(--ink);
  margin: 0 0 14px;
  line-height: 1.4;
  display: flex;
  gap: 10px;
  align-items: baseline;
}
.qa-id {
  display: inline-block;
  background: var(--surface-2);
  color: var(--ink-soft);
  font-family: ui-monospace, "SF Mono", SFMono-Regular, Menlo, Consolas, monospace;
  font-size: 11px;
  font-weight: 700;
  padding: 3px 8px;
  border-radius: 6px;
  flex-shrink: 0;
  letter-spacing: 0.02em;
}
.qa-answer {
  color: var(--ink-soft);
  font-size: 15px;
}
.qa-answer p { margin: 0 0 12px; }
.qa-answer p:last-child { margin-bottom: 0; }
.qa-answer ul, .qa-answer ol { margin: 8px 0 12px; padding-left: 22px; }
.qa-answer li { margin: 4px 0; }
.qa-answer strong { color: var(--ink); font-weight: 600; }
.qa-answer h1, .qa-answer h2, .qa-answer h3, .qa-answer h4 {
  font-size: 14px;
  font-weight: 700;
  margin: 14px 0 6px;
  color: var(--ink);
}
.qa-answer code {
  background: var(--surface-2);
  padding: 1px 5px;
  border-radius: 4px;
  font-size: 13px;
}

/* ---- Tables inside answers ---- */
.qa-answer table {
  width: 100%;
  border-collapse: collapse;
  margin: 14px 0;
  font-size: 14px;
  border: 1px solid var(--border);
  border-radius: 8px;
  overflow: hidden;
}
.qa-answer thead th {
  background: var(--surface-2);
  color: var(--ink);
  text-align: left;
  padding: 10px 14px;
  border-bottom: 1px solid var(--border);
  font-size: 11px;
  font-weight: 700;
  text-transform: uppercase;
  letter-spacing: 0.06em;
}
.qa-answer tbody td {
  padding: 10px 14px;
  border-bottom: 1px solid var(--border);
  color: var(--ink-soft);
  vertical-align: top;
}
.qa-answer tbody tr:last-child td { border-bottom: 0; }
.qa-answer tbody tr:nth-child(even) td { background: rgba(13, 27, 42, 0.02); }

.qa-refs {
  margin-top: 14px;
  padding-top: 12px;
  border-top: 1px dashed var(--border);
  font-size: 12px;
  color: var(--muted);
}
.qa-refs strong { color: var(--ink-soft); font-weight: 600; }

/* ---- Footer ---- */
.footer {
  text-align: center;
  color: var(--muted);
  font-size: 12px;
  padding: 36px 0 48px;
  border-top: 1px solid var(--border);
  margin-top: 32px;
}

/* ---- Responsive ---- */
@media (max-width: 820px) {
  .hero-inner { flex-direction: column; align-items: flex-start; }
  .hero h1 { font-size: 30px; }
  .kpis { grid-template-columns: repeat(2, 1fr); }
  .container { padding: 0 20px; }
  .section { padding: 24px; }
}
@media (max-width: 480px) {
  .kpis { grid-template-columns: 1fr; }
}

/* ---- Print ---- */
@media print {
  body { background: #fff; }
  .hero { background: #0d1b2a !important; -webkit-print-color-adjust: exact; print-color-adjust: exact; }
  .section, .kpi { box-shadow: none; break-inside: avoid; }
  .footer { padding: 16px 0; }
}
"""


def build_client_dashboard_html(rfp_name, questions, answers, vendor_name=VENDOR_NAME):
    """Self-contained HTML CxO dashboard for the prospect."""
    qmap = {q["id"]: q for q in questions}
    today = date.today()

    cats_present = {a.get("category") or "Other" for a in answers}
    unique_sources: set[str] = set()
    for a in answers:
        for s in a.get("sources", []):
            unique_sources.add(s)

    by_cat: dict[str, list[dict]] = {}
    for a in answers:
        by_cat.setdefault(a.get("category") or "Other", []).append(a)
    preferred = ["technical", "compliance", "pricing", "company-info"]
    ordered_cats = [c for c in preferred if c in by_cat] + [
        c for c in sorted(by_cat) if c not in preferred
    ]

    section_blocks = []
    for cat in ordered_cats:
        icon, title = CATEGORY_META.get(cat, ("📄", cat.replace("-", " ").title()))
        n = len(by_cat[cat])

        qa_blocks = []
        for ans in by_cat[cat]:
            q_id = _html.escape(str(ans.get("question_id", "?")))
            q_text = _html.escape(qmap.get(ans.get("question_id", ""), {}).get("text", ""))
            answer_html = _answer_to_html(ans.get("answer") or "")
            sources = ans.get("sources", [])
            refs_html = (
                f'<div class="qa-refs"><strong>References:</strong> '
                f'{_html.escape("; ".join(sources))}</div>'
                if sources else ""
            )
            qa_blocks.append(
                f'<div class="qa">'
                f'<div class="qa-question"><span class="qa-id">{q_id}</span>'
                f'<span>{q_text}</span></div>'
                f'<div class="qa-answer">{answer_html}</div>'
                f'{refs_html}'
                f'</div>'
            )

        section_blocks.append(
            f'<section class="section">'
            f'<div class="section-header">'
            f'<div class="icon">{icon}</div>'
            f'<h2>{_html.escape(title)}</h2>'
            f'<div class="count">{n} response{"s" if n != 1 else ""}</div>'
            f'</div>'
            f'{"".join(qa_blocks)}'
            f'</section>'
        )

    sections_html = "\n".join(section_blocks) or (
        '<section class="section"><p>No responses available.</p></section>'
    )

    rfp_name_e = _html.escape(rfp_name)
    vendor_e = _html.escape(vendor_name)
    date_full = today.strftime("%B %d, %Y")
    date_short = today.strftime("%b %Y")

    return f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Response to {rfp_name_e} | {vendor_e}</title>
<style>{_DASHBOARD_CSS}</style>
</head>
<body>
  <header class="hero">
    <div class="container hero-inner">
      <div>
        <div class="eyebrow">Proposal Response</div>
        <h1>Response to {rfp_name_e}</h1>
        <p class="sub">{vendor_e}'s answers to your request for proposal, organized by category. Each response is grounded in current product documentation, the certifications register, and customer success data.</p>
      </div>
      <div class="hero-meta">
        Prepared by <strong>{vendor_e}</strong><br>
        {date_full}
      </div>
    </div>
  </header>

  <main class="container">
    <section class="kpis">
      <div class="kpi"><div class="label">Questions Answered</div><div class="value">{len(answers)}</div></div>
      <div class="kpi"><div class="label">Categories Covered</div><div class="value">{len(cats_present)}</div></div>
      <div class="kpi"><div class="label">Source Documents</div><div class="value">{len(unique_sources)}</div><div class="hint">cited across responses</div></div>
      <div class="kpi"><div class="label">Prepared</div><div class="value">{date_short}</div></div>
    </section>

    {sections_html}
  </main>

  <footer class="footer container">
    {vendor_e} &middot; {date_full}
  </footer>
</body>
</html>
"""


# ------------------------------------------------------------
# Build the artifacts
# ------------------------------------------------------------
internal_report_md = build_internal_report(RFP_NAME, RFP_QUESTIONS, all_answers or [], _review)
client_dashboard_html = build_client_dashboard_html(RFP_NAME, RFP_QUESTIONS, all_answers or [])

final_output = {
    "rfp_name": RFP_NAME,
    "vendor": VENDOR_NAME,
    "generated": date.today().isoformat(),
    "total_questions": len(RFP_QUESTIONS),
    "answers": all_answers or [],
    "review": _review,
    "metadata": {
        "model": DEFAULT_MODEL,
        "knowledge_base_entries": len(KNOWLEDGE_BASE),
    },
}

out_dir = pathlib.Path("rfp_output")
out_dir.mkdir(exist_ok=True)
# UTF-8 explicitly — em-dashes and curly quotes in the answers will break
# on platforms whose default encoding isn't UTF-8.
(out_dir / "internal_report.md").write_text(internal_report_md, encoding="utf-8")
(out_dir / "client_response.html").write_text(client_dashboard_html, encoding="utf-8")
(out_dir / "rfp_response.json").write_text(
    json.dumps(final_output, indent=2, ensure_ascii=False), encoding="utf-8"
)

print(f"✓ Wrote 3 artifacts to ./{out_dir}/")
print(f"   - internal_report.md     ({len(internal_report_md):,} chars)")
print(f"   - client_response.html   ({len(client_dashboard_html):,} chars)")
print(f"   - rfp_response.json      ({len(final_output['answers'])} answers)")
print()
print(f"💡 Open ./{out_dir}/client_response.html in a browser to view the executive dashboard,")
print(f"   or print to PDF for circulation.")


✓ Wrote 3 artifacts to ./rfp_output/
   - internal_report.md     (12,311 chars)
   - client_response.html   (17,363 chars)
   - rfp_response.json      (5 answers)

💡 Open ./rfp_output/client_response.html in a browser to view the executive dashboard,
   or print to PDF for circulation.


---

## Part 8: ⭐ STRETCH — Eval Assertions

*This section is optional and unscored, but demonstrates rigor.*

Build assertions that validate your agent's output quality. Good evals test real failure modes, not just happy paths.

### What to Test

| Dimension | Example Assertion |
|-----------|-------------------|
| **Accuracy** | Q3 pricing answer contains "$18/seat/month" for 500 endpoints |
| **Source attribution** | Every answer has at least one source cited |
| **Consistency** | Q2 and Q5 don't contradict each other on certification dates |
| **Confidence calibration** | Questions with no KB match are flagged as low-confidence |
| **Edge cases** | What happens with an ambiguous or unanswerable question? |

In [12]:
# ============================================================
# ⭐ STRETCH: Write eval assertions
# ============================================================

def run_evals(answers: list[dict]) -> dict:
    """Run quality assertions against agent output."""
    results = {"passed": 0, "failed": 0, "details": []}

    for ans in answers:
        # Example assertion: every answer has sources
        has_sources = len(ans.get("sources", [])) > 0
        results["details"].append({
            "question": ans.get("question_id"),
            "assertion": "has_sources",
            "passed": has_sources,
        })
        if has_sources:
            results["passed"] += 1
        else:
            results["failed"] += 1

    # TODO: Add more assertions
    # - Check that pricing answer contains specific numbers
    # - Check that confidence is a valid value
    # - Check for cross-answer consistency
    # - Test an edge case question (e.g., a question with no KB match)

    return results


if all_answers:
    eval_results = run_evals(all_answers)
    print(f"Eval Results: {eval_results['passed']} passed, {eval_results['failed']} failed")
    for detail in eval_results["details"]:
        status = "✓" if detail["passed"] else "❌"
        print(f"  {status} {detail['question']}: {detail['assertion']}")
else:
    print("⚠️ Complete Part 5 first")

Eval Results: 5 passed, 0 failed
  ✓ Q1: has_sources
  ✓ Q2: has_sources
  ✓ Q3: has_sources
  ✓ Q4: has_sources
  ✓ Q5: has_sources


---

## 🏁 Wrap-Up

There's no formal demo, but if you built something you're proud of — a clever architecture, a useful analysis UI, an unusual approach to consistency review, a robust eval harness — you're strongly encouraged to grab a few minutes and show the room. Past builds we've seen and loved:

- RFP deck builders that hand off to slide generators
- Eval harnesses with assertions richer than the baseline in Part 8
- Analysis UIs that visualize per-answer confidence and source coverage
- Search UIs that take any RFP file and run the agent end-to-end
- Subagent routers that dispatch by category

### Worth reflecting on (for yourself, or for anyone you show your build to):

1. How did you handle the tool use loop? Sequential or batch?
2. What does your review step actually catch?
3. What broke (or would break) when you pointed it at the surprise RFP?
4. If you had another hour, what would you build first?